In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
from huggingface_hub import HfApi

REPO_ID = "vsevolod-nv/aiconf-butterfly-detection-all"

LABEL_FILES = {
    "butterfly": "data/butterflies.json",
    "bee": "data/bees.json",
    "beetle": "data/beetles.json",
    "flower": "data/flowers.json",
    "shrub": "data/shrubs.json",
}


/Users/mvsevolod/projects/aiconf-butterflies-segmentation/aiconf/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.concat(
    [
        pd.read_json(path).assign(taxon=taxon)
        for taxon, path in LABEL_FILES.items()
    ],
    ignore_index=True,
)

df = df[["taxon", "photo_id", "photo_url", "hard"]]

readme = f"""---
pretty_name: butterflies-detection
task_categories:
- image-classification
- image-segmentation
tags:
- biology
- insects
---

Columns:
- `taxon`
- `photo_id`
- `photo_url`
- `hard`

Dataset dedicated to further butterfly segmentation in the wild. The dataset contains photos of butterflies, bees, beetles, flowers, and shrubs, collected from iNaturalist. 

The `hard` column indicates whether the photo is considered a hard example for butterfly detection according to a few CV techniques.

Generated at: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}
Rows: {len(df)}
"""


In [ ]:
df.to_json("data/butterflies.json", orient="records", indent=4,)

<class 'pandas.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   taxon      1800 non-null   str  
 1   photo_id   1800 non-null   int64
 2   photo_url  1800 non-null   str  
 3   hard       1800 non-null   bool 
dtypes: bool(1), int64(1), str(2)
memory usage: 190.6 KB


In [3]:
api = HfApi()

with TemporaryDirectory() as tmp_dir:
    tmp_path = Path(tmp_dir)
    data_path = tmp_path / "data.jsonl"
    readme_path = tmp_path / "README.md"

    df.to_json(data_path, orient="records", lines=True, force_ascii=False)
    readme_path.write_text(readme, encoding="utf-8")

    api.create_repo(repo_id=REPO_ID, repo_type="dataset", exist_ok=True)
    api.upload_folder(
        repo_id=REPO_ID,
        repo_type="dataset",
        folder_path=str(tmp_path),
        commit_message="Upload butterflies-detection dataset manifest",
    )
